# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrabansal10/FlyRank_Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/rudrabansal10/FlyRank_Internship/43b468d73eba109085f02d01f3a59754d5356453/data/raw/content_refresh_anonymized.csv")


target = (df["trend_direction"] == "down").astype(int)
print("Positive-label rate:", target.mean())

Positive-label rate: 0.5420666666666667


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [17]:
import numpy as np

numeric_features = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
]

feature_frame = df[numeric_features + categorical_features].copy()

# Position 0 means "no position data", not first place.
feature_frame["has_position_data"] = (
    feature_frame["avg_position"] > 0).astype(int)

feature_frame.loc[feature_frame["avg_position"] == 0, "avg_position"] = np.nan

# Preserve systematic missingness before filling numeric fields.
for column in numeric_features:
    feature_frame[f"{column}_missing"] = feature_frame[column].isna().astype(int)

    feature_frame[column] = feature_frame[column].replace([np.inf, -np.inf], np.nan)

    feature_frame[column] = feature_frame[column].fillna(
                          feature_frame[column].median())

# Log transforms reduce the influence of extremely large traffic values.
for column in [
    "impressions_90d", "clicks_90d",
    "sessions_90d", "ai_sessions_90d",
]:
    feature_frame[f"log_{column}"] = np.log1p(feature_frame[column])

# Categorical blanks become an explicit "unknown" category.
for column in categorical_features:
    feature_frame[column] = feature_frame[column].fillna("unknown").astype(str)

X = pd.get_dummies(
    feature_frame,
    columns=categorical_features,
    dtype=int,
)

feature_columns = X.columns.tolist()

print("Feature-vector shape:", X.shape)
print("Target positive rate:", round(target.mean(), 4))
print("Remaining missing values:", int(X.isna().sum().sum()))

Feature-vector shape: (30000, 69)
Target positive rate: 0.5421
Remaining missing values: 0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The feature vector uses observable content, freshness, search-visibility, and engagement measures from the starter snapshot. Traffic fields are log-transformed where values are highly skewed. Missing numeric values are filled with the median only after creating a *_missing indicator, so the model can distinguish an unavailable measurement from a genuine zero. Missing categorical values are represented as unknown and one-hot encoded.
avg_position = 0 means position data is unavailable, not that the page ranks first. I convert those values to missing and add has_position_data.
These features are available in the starter snapshot before the model is run. However, this starter exercise uses a current-window proxy label, so it is not a genuine future-looking deployment setup. The full daily warehouse will be needed to create strictly earlier feature windows and later outcome windows

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [18]:
excluded = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "provider_used",
    "model_used",
]

# Prove excluded raw fields were not accidentally included.
leaked_raw_columns = sorted(set(excluded).intersection(feature_columns))
print("Excluded fields found in feature vector:", leaked_raw_columns)

assert leaked_raw_columns == [], "Leakage check failed."
assert X.shape[0] == len(target)
assert X.isna().sum().sum() == 0

# Deliberate leakage demonstration only — do NOT use this as a valid model result.
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

leaky_feature = df[["trend_pct"]].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(
    leaky_feature,
    target,
    test_size=0.2,
    random_state=42,
    stratify=target,
)

leaky_model = DecisionTreeClassifier(max_depth=2, random_state=42)
leaky_model.fit(X_train, y_train)
leaky_auc = roc_auc_score(y_test, leaky_model.predict_proba(X_test)[:, 1])

print("Leakage demonstration ROC-AUC using trend_pct:", round(leaky_auc, 3))
print("This number is invalid because trend_pct helps define the target.")

Excluded fields found in feature vector: []
Leakage demonstration ROC-AUC using trend_pct: 1.0
This number is invalid because trend_pct helps define the target.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Field/group | Why excluded |
|---|---|
| `content_id`, `client_id` | Identifiers; used only for joining and client-level splitting, never prediction. |
| `trend_direction` | Direct source of the proxy target. |
| `trend_pct` | Derived from the same trend used to define the proxy target. |
| Last/previous 30-day traffic fields | Closely tied to the periods used to create the proxy trend, so they could leak the answer. |
| `provider_used`, `model_used` | Metadata, not reliable performance signals. |
| Product flags/health scores, if available | Existing product decisions; using them would make the model circular. |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.